### RunnableParallel 데이터 병렬 조작

In [1]:
from dotenv import load_dotenv

load_dotenv()

from langchain_teddynote import logging

logging.langsmith("test0914")

LangSmith 추적을 시작합니다.
[프로젝트명]
test0914


In [2]:
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# 텍스트로부터 FAISS 벡터 저장소를 생성
vectorstore = FAISS.from_texts(
    ["Teddy is an AI enginner who loves programming!"],
    embedding= OpenAIEmbeddings()
)

# 벡터 저장소를 검색기로 사용
retriever = vectorstore.as_retriever()

# 템플릿을 정의
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""

# 템플릿으로부터 채팅 프롬프트를 생성
prompt = ChatPromptTemplate.from_template(template)

model = ChatOpenAI(model="gpt-4o-mini")

retrieval_chain = (
    {"context" : retriever, "question": RunnablePassthrough()}
    | prompt | model | StrOutputParser()
)

retrieval_chain.invoke("What is Teddy's occupation?")

"Teddy's occupation is an AI engineer."

### itemgetter 를 단축어롤 사용

In [3]:
from operator import itemgetter

from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# 텍스트로부터 FAISS 벡터 저장소를 생성합니다.
vectorstore = FAISS.from_texts(
    ["Teddy is an AI engineer who loves programming!"], embedding=OpenAIEmbeddings()
)
# 벡터 저장소를 검색기로 사용합니다.
retriever = vectorstore.as_retriever()

# 템플릿을 정의합니다.
template = """Answer the question based only on the following context:
{context}

Question: {question}

Answer in the following language: {language}
"""
# 템플릿으로부터 채팅 프롬프트를 생성합니다.
prompt = ChatPromptTemplate.from_template(template)

chain = (
    {
        "context" : itemgetter("question") | retriever,
        "question" : itemgetter("question"),
        "language" : itemgetter("language")
    } | prompt | ChatOpenAI(model="gpt-4o-mini") | StrOutputParser()
)

# 체인 호출
chain.invoke({"question" : "What is Teddy's occupation?" , "language" : "Korean"})

'테디의 직업은 AI 엔지니어입니다.'

### 병렬처리를 단계별로 이해


In [4]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableParallel
from langchain_openai import ChatOpenAI

model= ChatOpenAI()

capital_chain = (
    ChatPromptTemplate.from_template("{country} 의 수도는 어디입니까?")
    | model | StrOutputParser()
)

# 면적을 묻는 질문에 대한 체인
area_chain = (
    ChatPromptTemplate.from_template("{country} 의 면적은 얼마입니까?")
    | model | StrOutputParser()
)

map_chain = RunnableParallel(capital= capital_chain, area= area_chain)

map_chain.invoke({"country" : "나이지리아"})

{'capital': '나이지리아의 수도는 아부자입니다.', 'area': '나이지리아의 면적은 약 923,770 제곱 킬로미터 입니다.'}

In [5]:
# 수도를 묻는 질문에 대한 체인을 정의합니다.
capital_chain2 = (
    ChatPromptTemplate.from_template("{country1} 의 수도는 어디입니까?")
    | model
    | StrOutputParser()
)

# 면적을 묻는 질문에 대한 체인을 정의합니다.
area_chain2 = (
    ChatPromptTemplate.from_template("{country2} 의 면적은 얼마입니까?")
    | model
    | StrOutputParser()
)

# 병렬로 실행할 수 있는 RunnableParallel 객체 생성
map_chain2 = RunnableParallel(capital= capital_chain2, area= area_chain2)

# map_chain 호출
map_chain2.invoke({"country1" : "나이지리아", "country2" : "이란"})

{'capital': '나이지리아의 수도는 아부자입니다.', 'area': '이란의 면적은 약 1,648,195km² 입니다.'}

In [7]:
%%timeit

# 면저을 묻는 체인을 호출하고 실행 시간을 측정합니다.
area_chain.invoke({"country": "대한민국"})


1.49 s ± 436 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [8]:
%%timeit

# 수도를 묻는 체인을 호출하고 실행 시간을 측정합니다.
capital_chain.invoke({"country": "대한민국"})

1.08 s ± 434 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [9]:
%%timeit

# Parallel 하게 구성된 체인을 호출하고 실행 시간을 측정합니다.
map_chain.invoke({"country": "대한민국"})

1.86 s ± 399 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
